In [1]:
from protossl.datasets import HeedbECGDataset
from protossl.defines import HEEDB_TARGETS
from omegaconf import OmegaConf
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, balanced_accuracy_score, roc_auc_score

dataset_path = "/opt/gpudata/ecg/heedb"
arms = ["2D-partial", "2D-global", "1D-global"]
run_path = Path("/opt/gpu_working/steven/protoecgnet-heedb")
config_path = Path("/opt/gpu_working/steven/ProtoSSL/large-user-study/configs")
# dropping a branch from the head drops the labels only it covered, so the probs
# are scored against labels.yaml rather than all of HEEDB_TARGETS
fusion_path = run_path / "fusion-tuned/tune-fusion-classifier-drop-2D-global"
fusion_labels = list(
    OmegaConf.load(fusion_path / "labels.yaml").data.init_args.label_subset
)
# fusion label order is a subsequence of HEEDB_TARGETS, i.e. of the dataset columns
label_idxs = np.asarray([list(HEEDB_TARGETS).index(l) for l in fusion_labels])

In [ ]:
def stack_branched_arms():
    arm_labels = [l for arm in arms for l in OmegaConf.load(config_path / f"{arm}.yaml").data.init_args.label_subset]
    idxs = np.asarray([arm_labels.index(l) for l in fusion_labels]) # indices to normalize to fusion order

    ret = dict()
    for split in ["val", "test"] :
        arm_probs = {arm: np.load(run_path / arm / f"train-classifier/latest/{split}_probs.npy") for arm in arms}
        stacked_probs = np.concat([arm_probs[arm] for arm in arms], axis=1)
        ret[split] = stacked_probs[:, idxs]
    return ret

In [2]:
ds_phys = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    label_src="original_physician",
    heedb_split_type="by-label",
)

ds_phys_val = HeedbECGDataset(
    dataset_path=dataset_path,
    split="val",
    sampling_rate=100,
    label_src="original_physician",
    heedb_split_type="by-label",
)

ds_muse = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    label_src="original_muse",
    heedb_split_type="by-label",
)

================get_heedb_metadata=================
using by-label splits
reading HEEDB metadata from on-disk cache: /home/songs1/.cache/protossl_cache/abac20ad.csv
=================make_heedb_labels=================
reading HEEDB labels from on-disk cache: /home/songs1/.cache/protossl_cache/51ff3555.npy
replaying below stats from cache:
Of 453512 ECGs and 11607259 annotations, 453512 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly
=================make_heedb_labels=================
reading HEEDB labels from on-disk cache: /home/songs1/.cache/protossl_cache/c6af0872.npy
replaying below stats from cache:
Of 453512 ECGs and 11607259 annotations, 453512 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly
=================make_h

In [3]:
y_true = ds_phys.labels.numpy()[:, label_idxs]
y_true_val = ds_phys_val.labels.numpy()[:, label_idxs]
y_pred_muse = ds_muse.labels.numpy()[:, label_idxs]

In [4]:
def compute_threshold(_y_true, _y_prob):
    # NOTE: should be computed on either train/val set, NOT test
    fpr, tpr, thresholds = roc_curve(_y_true, _y_prob)
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    return optimal_threshold

In [5]:
# temp = stack_branched_arms()
# y_prob_branched, y_prob_branched_val = temp["test"], temp["val"]
y_prob_fusion = np.load(fusion_path / "test_probs.npy")
y_prob_fusion_val = np.load(fusion_path / "val_probs.npy")
assert y_prob_fusion.shape[1] == len(fusion_labels)

# y_pred_branched = np.zeros_like(y_pred_muse)
y_pred_fusion = np.zeros_like(y_pred_muse)
accuracy_results = []
roc_results = []
for i, label in enumerate(fusion_labels):
    # branched_threshold = compute_threshold(y_true_val[:, i], y_prob_branched_val[:, i])
    # y_pred_branched[:, i] = (y_prob_branched[:, i] > branched_threshold).astype(int)
    fusion_threshold = compute_threshold(y_true_val[:, i], y_prob_fusion_val[:, i])
    y_pred_fusion[:, i] = (y_prob_fusion[:, i] > fusion_threshold).astype(int)
    label_acc_results = {
        "label": label,
        "muse: BalAcc": balanced_accuracy_score(y_true[:, i], y_pred_muse[:, i]),
        # "branched: BalAcc": balanced_accuracy_score(y_true[:, i], y_pred_branched[:, i]),
        "fusion: BalAcc": balanced_accuracy_score(y_true[:, i], y_pred_fusion[:, i]),
    }
    label_roc_results = {
        "label": label,
        # NOTE: can't compute muse roc given no probs
        # "branched: AUROC": roc_auc_score(y_true[:, i], y_prob_branched[:, i]),
        "fusion: AUROC": roc_auc_score(y_true[:, i], y_prob_fusion[:, i]),
    }
    accuracy_results.append(label_acc_results)
    roc_results.append(label_roc_results)

In [6]:
roc_df = pd.DataFrame(roc_results)
idx = len(roc_df)
roc_df.loc[idx, "label"] = "Macro Average"
for c in [c for c in roc_df.columns if c != "label"]:
    roc_df.loc[idx, c] = roc_df[c].mean()

with pd.option_context("display.precision", 3):
    display(roc_df)

,label,fusion: AUROC
0,ANTERIOR INFARCT,0.953
1,ATRIAL FIBRILLATION,0.959
2,ATRIAL FLUTTER,0.953
3,ATRIAL-PACED RHYTHM,0.986
4,INCOMPLETE RIGHT BUNDLE BRANCH BLOCK,0.962
5,INFERIOR INFARCT,0.976
6,LATERAL INFARCT,0.957
7,LEFT BUNDLE BRANCH BLOCK,0.987
8,PREMATURE ATRIAL COMPLEXES,0.912
9,PREMATURE VENTRICULAR COMPLEXES,0.896


In [7]:
acc_df = pd.DataFrame(accuracy_results)
idx = len(acc_df)
acc_df.loc[idx, "label"] = "Macro Average"
for c in [c for c in acc_df.columns if c != "label"]:
    acc_df.loc[idx, c] = acc_df[c].mean()

with pd.option_context("display.precision", 3):
    display(acc_df)

,label,muse: BalAcc,fusion: BalAcc
0,ANTERIOR INFARCT,0.949,0.893
1,ATRIAL FIBRILLATION,0.921,0.905
2,ATRIAL FLUTTER,0.858,0.888
3,ATRIAL-PACED RHYTHM,0.873,0.943
4,INCOMPLETE RIGHT BUNDLE BRANCH BLOCK,0.910,0.908
5,INFERIOR INFARCT,0.972,0.918
6,LATERAL INFARCT,0.973,0.909
7,LEFT BUNDLE BRANCH BLOCK,0.940,0.958
8,PREMATURE ATRIAL COMPLEXES,0.910,0.850
9,PREMATURE VENTRICULAR COMPLEXES,0.929,0.831
